In [ ]:
# Install Cadabra if necessary, and enable Cadabra input in this notebook.
try:
    import cadabra2
except ImportError:
    %pip install -q cadabra2
%load_ext cadabra2_jupyter

# Bosonic string oscillator algebra

The computation below shows how you can do oscillator algebra for the (open) bosonic string.
We illustrate this by the classic computation which shows that you can construct a state
which satisfies all the constraints but has negative norm when $d>26$. This is not a terribly difficult
computation with pen-and-paper, but it is tricky to do with many other computer algebra systems (and
if they handle it, their output will not look as nice as what you see below).

In [ ]:
\vac::LaTeXForm("|k\rangle");
\state{A??}::LaTeXForm("|", A??, "\rangle");

In [ ]:
def post_process(ex):
   substitute(ex, $\sqrt{2} \sqrt{2} -> 2$)
   collect_terms(ex)

We first need to define the various objects, mostly non-commuting operators.

In [ ]:
\alpha_{n}^{\mu}::SelfNonCommuting;
{\alpha_{n}^{\mu}, \vac}::NonCommuting;
{\mu,\nu,\rho}::Integer(0..d-1).
{\mu,\nu,\rho}::Indices.
{m,n}::Integer.
{m,n}::Symbol.
\delta{#}::KroneckerDelta.
\delta{#}::Diagonal.

The trial state is given by $|\chi\rangle = |\chi_1\rangle + A |\chi_2\rangle + B |\chi_3\rangle$ with the three terms
given by the following expressions:

In [ ]:
chi1:= \alpha_{-1}^{\mu} \alpha_{-1}^{\mu} \vac;
chi2:= \alpha_{0}^\mu \alpha_{-2}^\mu \vac;
chi3:= \alpha_{0}^{\mu} \alpha_{-1}^{\mu} \alpha_{0}^{\nu} \alpha_{-1}^{\nu})\vac;

The main ingredient to verify that the $L_1$ and $L_2$ Virasoro constraints hold is, of course, the commutation relations
between the $\alpha_{m}^\mu$ operators. What we want to do is commute all annihilation operators to the right until they hit
the momentum eigenstate $|k\rangle$. Note that in the rule below, you need to write the indices on $\delta_{m, -n}$ using two separate subscripts, otherwise
you get $m-n$ as index.

In [ ]:
rl:= { \alpha_{m}^{\mu} \alpha_{n}^{\nu} | m > n -> \alpha_{n}^{\nu} \alpha_{m}^{\mu} +  m\delta_{m}_{-n} \delta^{\mu\nu},
       \alpha_{m}^{\mu} \vac | m > 0 -> 0, 
       \alpha_{0}^{\mu} \vac -> \sqrt{2} k^{\mu} \vac, k^{\mu} k^{\mu} -> -1 };

Obviously we need to do this substitution a number of times, and distribute any factors over sums. We will see
shortly how to automate this with `converge`.

## Imposing the constraints

The first thing to do is to impose the Virasoro constraints on the state $|\chi\rangle$, which will pin down the relative coefficients of the three terms in that state. The $L_m$ Virasoro generators are given by infinite sums. The function below constructs part of that sum explicitly. Of course, you would be a bit more clever when doing things with pen and paper:

In [ ]:
def L(m):
	""" Construct the m-th Virasoro generator in terms of creation/annihilation operators """
	ins=''
	for n in range(-10,10):
		if n!=-10:
			ins+='+ '
		ins += r'1/2 \alpha_{'+str(m-n)+r'}^{\mu} \alpha_{'+str(n)+r'}^{\mu}'
	ex=Ex(ins)
	return ex

Here is what $L_0$ looks like; the other operators are similar:

In [ ]:
L0=L(0);
L1=L(1)
L2=L(2)

The $L_1$ operator acting on the three terms in the trial state gives the following three terms, which we will need to reduce
using the commutation relations (2nd and 3rd term suppressed; you get the idea).

In [ ]:
L1c1:= @(L1) @(chi1);
L1c2:= @(L1) @(chi2).
L1c3:= @(L1) @(chi3).

We can now work out what is the action of a Virasoro operator on a state by repeatedly substituting the 
commutation relations of the $\alpha_{m}^{\mu}$ and simplifying, until the result no longer changes.
Below, we stick that logic into a function `act` which you can apply on an expression.

In [ ]:
def act(ex):
	converge(ex):
		distribute(ex)
		substitute(ex, rl)
		distribute(ex)
		eliminate_kronecker(ex)
		rename_dummies(ex)

	return ex

Here is what happens when you apply the commutator rules to the three terms in the state $L_1 |\chi\rangle$:

In [ ]:
act(L1c1);

In [ ]:
act(L1c2);

In [ ]:
act(L1c3);

The procedure works the same for $L_2|\chi\rangle$:

In [ ]:
L2c1:= @(L2) @(chi1).
L2c2:= @(L2) @(chi2).
L2c3:= @(L2) @(chi3).

In [ ]:
act(L2c1);

In [ ]:
act(L2c2);

In [ ]:
act(L2c3);

The constraints $L_1 |\chi\rangle=0$ and $L_2 |\chi\rangle=0$ thus reduce to

In [ ]:
tst1:=@(L1c1) + A @(L1c2) + B @(L1c3) = 0:
sort_product(tst1);

In [ ]:
tst2:=@(L2c1) + A @(L2c2) + B @(L2c3) = 0;

In [ ]:
factor_in(tst1, $d,A,B$);

In [ ]:
factor_in(tst2, $d,A,B$);

We then just need to solve this system of linear equations using SymPy's `linsolve`,

In [ ]:
from cdb.sympy.solvers import *

In [ ]:
substitute(tst1,$\vac->1, \alpha{#}->1, k^\mu=1$)
substitute(tst2,$\vac->1, \alpha{#}->1$)

In [ ]:
linsolve($@(tst1),@(tst2)$,$A,B$);

These are thus (indeed) solved by $A=\frac{d-1}{5}$ and $B=\frac{d+4}{10}$.

## Computing the norm

Having pinned down the coefficients $A$ and $B$, we now want to compute the norm of the state $|\chi\rangle$ (to be added).